# 01 — Introducción a Supervision

## ¿Qué vamos a construir hoy?

Al final de este notebook tendrás esto funcionando:

```
Imagen → Modelo YOLO → sv.Detections → Imagen anotada
```

**Aprenderás a:**
- Entender qué problema resuelve Supervision y por qué existe
- Usar `sv.Detections` — el objeto central de la librería
- Ejecutar tu primer pipeline completo: detección + anotación

**Tiempo estimado:** 30 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## El problema que resuelve Supervision

Imagina que quieres construir una aplicación de visión computacional.
Eliges un modelo (YOLO, SAM, Detectron2...) y cada uno te da sus resultados
en un formato diferente e incompatible con los demás.

**Supervision es como un adaptador de enchufes de viaje:**
cada modelo tiene su propio "conector" (formato de salida).
Supervision convierte todos al mismo formato (`sv.Detections`).
La información de la detección es la misma — solo cambia el conector.

```
YOLO results  ──┐
SAM results   ──┼──► sv.Detections ──► Annotators, Trackers, Zones
Transformers  ──┘
```

In [ ]:
# Instalar las librerías necesarias para este notebook
# supervision: la librería que vamos a aprender
# ultralytics: el framework que provee el modelo YOLO
# Descomenta la línea siguiente si estás ejecutando en un entorno nuevo:
!pip install supervision ultralytics

In [ ]:
import supervision as sv
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"Supervision versión: {sv.__version__}")

## Descargando la imagen de prueba

Usaremos una imagen pública con varios objetos para que las detecciones sean interesantes.

In [ ]:
import urllib.request

Path("assets").mkdir(exist_ok=True)

urllib.request.urlretrieve(
    "https://ultralytics.com/images/bus.jpg",
    "assets/bus.jpg"
)

image = cv2.imread("assets/bus.jpg")
print(f"Imagen cargada: {image.shape}")
# image.shape → (alto, ancho, canales)
# Los canales son BGR en OpenCV (azul, verde, rojo) — no RGB como en matplotlib

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Imagen de prueba")
plt.show()

## ¿Qué es un bounding box?

Un **bounding box** (caja delimitadora) es el rectángulo más pequeño que enmarca
completamente un objeto detectado en una imagen.

Se representa con cuatro coordenadas en formato `xyxy`:
```
[x_izquierda, y_arriba, x_derecha, y_abajo]
      x1          y1        x2        y2
```

El origen `(0, 0)` está en la **esquina superior-izquierda** de la imagen.
Las coordenadas se miden en píxeles.

```
(x1, y1)  ┌─────────────┐
          │             │
          │   objeto    │
          │             │
          └─────────────┘ (x2, y2)
```

Supervision almacena todos los bounding boxes en `detections.xyxy`:
una tabla NumPy donde **cada fila es un objeto** y las columnas son `[x1, y1, x2, y2]`.

## ¿Qué es Supervision?

[**Supervision**](https://supervision.roboflow.com/latest/) es una librería Python
de Roboflow para trabajar con visión computacional de forma **agnóstica al framework**.

El problema que resuelve: cada modelo (YOLO, SAM, Detectron2, Transformers) devuelve
sus resultados en un formato diferente e incompatible con los demás.
Supervision convierte todos esos formatos al mismo objeto central: `sv.Detections`.

Una vez que tienes un `sv.Detections`, puedes usar cualquier anotador, tracker o zona
**sin importar qué modelo generó la detección** — el código no cambia.

```
YOLO results  ──┐
SAM results   ──┼──► sv.Detections ──► Annotators, Trackers, Zones
Transformers  ──┘
```

> **Documentación oficial:** https://supervision.roboflow.com/latest/

## ¿Qué es YOLO?

**YOLO** (*You Only Look Once*) es una familia de modelos de detección de objetos
en tiempo real. Su nombre describe su arquitectura: analiza la imagen completa
en una sola pasada (*look once*), lo que lo hace extremadamente rápido.

- **YOLOv8 (Ultralytics):** la versión moderna más usada, con API sencilla de Python.
  → https://yolov8.com/
- **YOLO original (Darknet):** el trabajo académico fundacional de Joseph Redmon.
  → https://pjreddie.com/darknet/yolo/

El sufijo del archivo indica el tamaño del modelo:

| Sufijo | Tamaño aprox. | Velocidad | Precisión |
|--------|--------------|-----------|-----------|
| `n` (nano) | ~6 MB | +++  | +   |
| `s` (small) | ~22 MB | ++ | ++ |
| `m` (medium) | ~52 MB | + | +++ |

En este curso usamos `yolov8n.pt` para maximizar velocidad en CPU.
Si tienes GPU, prueba `yolov8s.pt` o `yolov8m.pt` para mayor precisión.

## El dataset COCO y su relación con YOLO

Los modelos YOLOv8 pre-entrenados aprenden a detectar objetos del dataset
**COCO** (*Common Objects in Context*): ~330 000 fotografías del mundo real con
80 categorías etiquetadas a mano.

→ https://cocodataset.org/#home

Por eso cuando ejecutas YOLO sobre cualquier imagen, las clases disponibles son
siempre las mismas 80 categorías COCO: personas, vehículos, animales, mobiliario, etc.

Las más frecuentes en los ejemplos de este curso:

| `class_id` | nombre |
|-----------|--------|
| 0 | person |
| 2 | car |
| 5 | bus |
| 7 | truck |

`results.names` es el diccionario que traduce `class_id` → nombre de clase COCO.

## Paso 1: Cargar el modelo YOLO

YOLOv8 es un modelo entrenado para detectar 80 tipos de objetos (personas, autos, animales, etc.).
La letra al final indica el tamaño: `n`=nano (más rápido), `s`=small, `m`=medium, `l`=large, `x`=extra-large.

In [ ]:
model = YOLO("yolov8n.pt")
# yolov8n.pt se descarga automáticamente (~6 MB) la primera vez
# Supervision funciona igual con cualquier tamaño de modelo — solo cambia esta línea

## Paso 2: Detectar y convertir a sv.Detections

Aquí ocurre la "traducción": los resultados de YOLO se convierten al formato universal de Supervision.

In [ ]:
results = model(image)[0]
# model() acepta una imagen y devuelve UNA LISTA de resultados
# [0] toma el primer (y único) resultado — siempre necesario aunque proceses una sola imagen

detections = sv.Detections.from_ultralytics(results)
# from_ultralytics() realiza la traducción al formato universal sv.Detections
# Si usáramos otro framework, el código de aquí en adelante sería IDÉNTICO

## Pausa y observa: ¿Qué hay dentro de sv.Detections?

`sv.Detections` es como una tabla con una fila por objeto detectado.
Cada columna describe un aspecto de la detección.

In [ ]:
print(f"Número de objetos detectados: {len(detections)}")

print(f"\n--- xyxy: coordenadas del bounding box ---")
print("Formato: [x_izquierda, y_arriba, x_derecha, y_abajo]")
print(detections.xyxy)

print(f"\n--- confidence: certeza del modelo (0 = inseguro, 1 = muy seguro) ---")
print(detections.confidence)

print(f"\n--- class_id: número de la categoría detectada ---")
print(detections.class_id)

print(f"\n--- Traducción de class_id a nombre ---")
for class_id in sorted(set(detections.class_id)):
    print(f"  Clase {class_id}: {results.names[class_id]}")

## Paso 3: Anotar la imagen

Los `Annotators` dibujan las detecciones sobre una imagen.
Se encadenan: la salida de uno es la entrada del siguiente.

In [ ]:
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

labels = [
    f"{results.names[class_id]} {conf:.0%}"
    for class_id, conf in zip(detections.class_id, detections.confidence)
]

# image.copy() es IMPORTANTE: evita modificar la imagen original
# Sin .copy(), los experimentos siguientes verán la imagen ya anotada
annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Pipeline completo: detección + anotación con Supervision")
plt.show()

## 🔧 Exploración interactiva

### Experimento 1: Umbral de confianza

¿Qué pasa cuando le pedimos al modelo que sea más estricto?

In [ ]:
results_estricto = model(image, conf=0.8)[0]
detections_estricto = sv.Detections.from_ultralytics(results_estricto)

print(f"Con conf=0.5 (por defecto): {len(detections)} objetos detectados")
print(f"Con conf=0.8 (estricto):    {len(detections_estricto)} objetos detectados")

# 💭 Reflexión: ¿Por qué detecta menos objetos con confianza más alta?
# El modelo tiene más certeza en objetos grandes y bien visibles.
# Subir el umbral elimina las detecciones "dudosas".

### Experimento 2: Inspeccionar una detección específica

In [ ]:
primera = detections[0]
# sv.Detections soporta indexación igual que una lista de Python
# primera es otro sv.Detections con un solo objeto

x1, y1, x2, y2 = primera.xyxy[0]
print(f"Primera detección:")
print(f"  Clase:    {results.names[primera.class_id[0]]}")
print(f"  Confianza: {primera.confidence[0]:.1%}")
print(f"  Posición:  esquina superior-izquierda ({x1:.0f}, {y1:.0f})")
print(f"             esquina inferior-derecha   ({x2:.0f}, {y2:.0f})")
print(f"  Tamaño:    {x2-x1:.0f} px de ancho × {y2-y1:.0f} px de alto")

# 💭 Reflexión: Las coordenadas se miden en píxeles desde la esquina superior-izquierda.
# ¿Puedes estimar a qué objeto de la imagen corresponde esta detección?

### Experimento 3: Cambiar el modelo

El código de Supervision no cambia al cambiar el modelo. Solo cambia la línea de YOLO.

In [ ]:
model_s = YOLO("yolov8s.pt")  # ~22 MB — más preciso, más lento
results_s = model_s(image)[0]
detections_s = sv.Detections.from_ultralytics(results_s)

print(f"yolov8n (nano):  {len(detections)} objetos")
print(f"yolov8s (small): {len(detections_s)} objetos")

# 💭 Reflexión: ¿Detectan los mismos objetos? ¿Con la misma confianza?
# El modelo más grande suele detectar más objetos pequeños u ocluidos.

## 🚀 Reto de extensión

**Tarea:** Ejecuta el pipeline completo con una imagen tuya o descargada de internet.

1. Descarga una imagen con `urllib.request.urlretrieve("URL", "mi_imagen.jpg")`
2. Cárgala con `cv2.imread("mi_imagen.jpg")`
3. Ejecuta el mismo pipeline (model → from_ultralytics → annotate)
4. ¿Qué objetos detecta? ¿Hay detecciones incorrectas?

**Pista:** Si no sabes qué imagen usar:
```python
urllib.request.urlretrieve(
    "https://ultralytics.com/images/zidane.jpg",
    "assets/zidane.jpg"
)
```

In [ ]:
# Escribe tu solución aquí
# mi_imagen = cv2.imread("mi_imagen.jpg")

## 💾 Guardar predicciones en JSON

Además de las imágenes anotadas, puedes guardar las predicciones en formato JSON
en la carpeta `assets/`. Útil para análisis posterior sin necesitar re-ejecutar el modelo.

In [ ]:
import json

def detections_to_dict(detections, class_names=None):
    """Convierte sv.Detections a un dict JSON-compatible."""
    return {
        "xyxy":        detections.xyxy.tolist(),
        "confidence":  detections.confidence.tolist() if detections.confidence is not None else None,
        "class_id":    detections.class_id.tolist()   if detections.class_id   is not None else None,
        "class_names": [class_names[c] for c in detections.class_id]
                       if (class_names and detections.class_id is not None) else None,
    }

resultado = detections_to_dict(detections, class_names=results.names)

with open("assets/predicciones.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)

print("Guardado: assets/predicciones.json")
print(json.dumps(resultado, indent=2))